# Sampling

In [ ]:
import pandas as pd
from sampling import balanced_weighted_sample
import numpy as np
import scipy.stats as ss

In [ ]:
df = pd.read_csv("../csv_data/stratification_data.csv")
sample_size = 500
balance_cols = [
    "diagnosis",
    "super_stain",
    "year",
    "lab_name",
    "patho_scanner_manufacturer",
]

sampled_df_02 = balanced_weighted_sample(
    df,
    balance_cols=balance_cols,
    N=sample_size,
    power=0.2,
)

sampled_df_04 = balanced_weighted_sample(
    df,
    balance_cols=balance_cols,
    N=sample_size,
    power=0.4,
)

sampled_df_06 = balanced_weighted_sample(
    df,
    balance_cols=balance_cols,
    N=sample_size,
    power=0.6,
)

sampled_df_08 = balanced_weighted_sample(
    df,
    balance_cols=balance_cols,
    N=sample_size,
    power=0.8,
)

sampled_df_02.to_csv("../csv_data/balanced_weighted_sample_02.csv", index=False)
sampled_df_04.to_csv("../csv_data/balanced_weighted_sample_04.csv", index=False)
sampled_df_06.to_csv("../csv_data/balanced_weighted_sample_06.csv", index=False)
sampled_df_08.to_csv("../csv_data/balanced_weighted_sample_08.csv", index=False)

In [ ]:
all_samples = [sampled_df_02, sampled_df_04, sampled_df_06, sampled_df_08]
all_samples_names = ["sampled_df_02", "sampled_df_04", "sampled_df_06", "sampled_df_08"]

### Choice of the sample

In [ ]:
selected_sampled_df = (
    sampled_df_08  # Change this to select a different sample for validation
)

# Validation

In [ ]:
cols = ["diagnosis", "super_stain", "year", "lab_name", "patho_scanner_manufacturer"]
from sampling_validation_utils import (
    marginal_balance_df,
    tvd,
    normalized_entropy,
    combination_diversity,
    joint_entropy,
    coverage,
    cramers_v,
)

### 1) Single modality comparison

In [ ]:
from itables import init_notebook_mode, show

init_notebook_mode(all_interactive=True)
balance_tables = marginal_balance_df(df, selected_sampled_df, cols)
# show(balance_tables)

#### a) Distance between initial and sampled distributions

***Total Variation Distance***: proportion of the mass that should be moved in the sampled distribution to reproduce the initial distribution.

In [ ]:
for i, sampled_df in enumerate(all_samples):
    print(all_samples_names[i])
    for col in cols:
        p = df[col].value_counts(normalize=True)
        q = sampled_df[col].value_counts(normalize=True)
        print(col, f"{tvd(p, q):.2f}")
    print("\n")

#### b) Entropy per variable (original and sampled)

Quantifies the uncertainty: higher means closer to uniform distribution, lower means closer to unimodal distribution

In [ ]:
for i, sampled_df in enumerate(all_samples):
    print(all_samples_names[i])
    for col in cols:
        print(
            col,
            f"{normalized_entropy(df[col]):.2f}, {normalized_entropy(sampled_df[col]):.2f}",
        )
    print("\n")

### 2) Joint modality diversity:

#### a) Unique combinations

In [ ]:
print(
    "proportion of unique combinations in initial df:",
    f"{combination_diversity(df, cols):.2f}",
)
for i, sampled_df in enumerate(all_samples):
    print(
        "proportion of unique combinations in",
        all_samples_names[i],
        ":",
        f"{combination_diversity(sampled_df, cols):.2f}",
    )

#### b) Joint entropy

Entropy where the modalities are the unique combinations.

In [ ]:
print("joint entropy in initial df:", f"{joint_entropy(df, cols):.2f}")
for i, sampled_df in enumerate(all_samples):
    print(
        "joint entropy in",
        all_samples_names[i],
        ":",
        f"{joint_entropy(sampled_df, cols):.2f}",
    )

#### c) Original space coverage

In [ ]:
for i, sampled_df in enumerate(all_samples):
    print(
        "proportion of original combinations preserved with",
        all_samples_names[i],
        ":",
        f"{coverage(df, sampled_df, cols):.2f}",
    )

### 3) Pairwise interaction (original and sampled)

Should be similar. An increased value means an artificial correlation was introduced, a lower one means an existing one was erased.

In [ ]:
for i, sampled_df in enumerate(all_samples):
    print(all_samples_names[i])
    for i, c1 in enumerate(cols):
        for c2 in cols[i + 1 :]:
            print(
                c1,
                c2,
                f"{cramers_v(df[c1], df[c2]):.2f}",
                f"{cramers_v(sampled_df[c1], sampled_df[c2]):.2f}",
            )
    print("\n")

### 4) Glomeruli count

In [ ]:
cresc_freq_sampled = selected_sampled_df["number_glom_crescent"].sum() / len(
    selected_sampled_df
)
cresc_freq_initial = df["number_glom_crescent"].sum() / len(df)
print(f"Crescent glomeruli per slide in sampled data: {cresc_freq_sampled:.2f}")
print(f"Crescent glomeruli per slide in initial data: {cresc_freq_initial:.2f}")

In [ ]:
glom_freq_sampled = selected_sampled_df["number_glom"].sum() / len(selected_sampled_df)
glom_freq_initial = df["number_glom"].sum() / len(df)
print(f"Total glomeruli per slide in sampled data: {glom_freq_sampled:.2f}")
print(f"Total glomeruli per slide in initial data: {glom_freq_initial:.2f}")

In [ ]:
proportion_cresc = df["number_glom_crescent"].div(df["number_glom"].replace(0, np.nan))
proportion_cresc_sampled = selected_sampled_df["number_glom_crescent"].div(
    selected_sampled_df["number_glom"].replace(0, np.nan)
)
proportion_cresc = proportion_cresc[proportion_cresc != 0]
proportion_cresc_sampled = proportion_cresc_sampled[proportion_cresc_sampled != 0]

In [ ]:
print("proportion of the glomeruli that are crescent in sampled data:")
print(proportion_cresc_sampled.describe())
print("proportion of the glomeruli that are crescent in initial data:")
print(proportion_cresc.describe())

In [ ]:
print(
    f'{selected_sampled_df["number_glom_crescent"].sum()} crescent glomeruli in sampled data from {df["number_glom_crescent"].sum()} initially'
)

# Visual validation

### Value counts per variable of interest

In [ ]:
from sampling_validation_utils import plot_value_counts

plot_value_counts(selected_sampled_df, cols)

### Absolute frequency difference per variable

In [ ]:
from sampling_validation_utils import (
    plot_marginal_abs_diff_plotly,
    plot_marginal_distributions_plotly,
)

"""
for i in range(len(all_samples)):
    plot_marginal_distributions(df, all_samples[i], cols, all_samples_names[i])
    plot_marginal_abs_diff(df, all_samples[i], cols, all_samples_names[i])
"""

plot_marginal_abs_diff_plotly(df, selected_sampled_df, cols, "sampled data")

### Relative frequency difference per variable

In [ ]:
from sampling_validation_utils import plot_marginal_relative_diff_plotly

plot_marginal_relative_diff_plotly(df, selected_sampled_df, cols, "sampled data")

# Prepare for use in QuPath

## Save data with full access path

In [ ]:
def recreate_full_path(row, patient_group):
    year = row["ANON_name"].split("_")[0]
    suffix = row["file_name"].split(".")[-1]
    return f"\\\\ihelse.net\\Forskning\\hbe\\2023-517496\\RegistryWSIs\\{patient_group}\\{year}_anon\\{row['ANON_name']}.{suffix}"

In [ ]:
import pandas as pd

df = (
    selected_sampled_df.copy()
)  # pd.read_csv("../csv_data/balanced_weighted_sample_08.csv")
kb_df = pd.read_csv("../csv_data/KB_csv.csv")
nkbr_df = pd.read_csv("../csv_data/NKBR_csv.csv")
kb_df = kb_df[["ANON_name", "file_name"]]
kb_df["full_path"] = kb_df.apply(
    lambda row: recreate_full_path(row, "Kidney biopsies"), axis=1
)
nkbr_df = nkbr_df[["ANON_name", "file_name"]]
nkbr_df["full_path"] = nkbr_df.apply(
    lambda row: recreate_full_path(row, "The Norwegian Kidney Biopsy Registry"), axis=1
)

together_df = pd.concat([kb_df, nkbr_df], ignore_index=True)
df = df.merge(together_df, left_on="wsi_anon_name", right_on="ANON_name", how="inner")

**!! Define the name of the place to store the produced list in the following cell.**

In [ ]:
list_name = "final_list"
df.to_csv(f"../csv_data/{list_name}.csv", index=False)

## Save file paths as list

### Preparation for group inclusion in QuPath project

In [ ]:
def save_list_to_file(lst, file_path):
    content_to_save = "\n".join(lst)
    try:
        with open(file_path, "w") as file:
            file.write(content_to_save)
        print(f"List successfully saved to {file_path}")
    except Exception as e:
        print(f"An error occurred while saving the list: {e}")

In [ ]:
final_list_df = pd.read_csv(f"../csv_data/{list_name}.csv")
final_list = final_list_df["full_path"].tolist()
file_path = list_name + ".txt"
save_list_to_txt(final_list, file_path)